# 35 - LLM judge ensemble for the pooled evaluation set

Judges each trustworthy pooled candidate (from notebook 34) with every configured LLM judge (Ollama, OpenAI, Claude, whichever have credentials set below), on a 3-level relevance scale: 2 = highly relevant, 1 = partially relevant, 0 = not relevant.

Where judges agree, that becomes a high-confidence gold label. Where they disagree, the candidate goes into a separate manual-review file instead of being silently averaged, real human judgment (yours) breaks the tie, not an automatic vote.

Saved after every single candidate, not just every query, same reasoning as the API oversample notebook: paid judges cost real money per call, an interruption should never mean redoing work already paid for.

**Run the pilot cell first** (a handful of queries) to sanity-check cost and agreement rate before scaling to the full pool.

In [2]:
import os, json, time
import pandas as pd
import requests
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(override=True)

OUTPUT_DIR = Path("result/35_llm_judge_ensemble")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_PATH = OUTPUT_DIR / "judge_cache.json"

PILOT_N_QUERIES = 5  # set to None to run the full 101-query pool once the pilot looks good

# Matches run.sh's convention: Ollama is started fresh inside a SLURM job
# (module load cs/ollama/0.5.11, then `ollama serve` on 127.0.0.1:11435), and
# run.sh writes OLLAMA_BASE_URL (not OLLAMA_HOST) to .env for the duration of
# that job. This notebook needs to be run the same way, with Ollama already
# started on that port in the current job/session, or OLLAMA_BASE_URL pointed
# at wherever else it's running.
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://127.0.0.1:11435")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3.1")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")


def ollama_available():
    try:
        r = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=3)
        return r.status_code == 200
    except requests.exceptions.RequestException:
        return False


ENABLED_JUDGES = {
    "ollama": ollama_available(),
    "openai": OPENAI_API_KEY is not None,
    "claude": ANTHROPIC_API_KEY is not None,
}
print("Judges enabled:", ENABLED_JUDGES)
if sum(ENABLED_JUDGES.values()) < 2:
    print("WARNING: fewer than 2 judges enabled -- agreement/disagreement voting needs at least 2 to mean anything.")
    print("  Ollama: start it the same way run.sh does (module load cs/ollama/0.5.11, ollama serve on 127.0.0.1:11435), or set OLLAMA_BASE_URL to wherever it's running.")
    print("  OpenAI: add OPENAI_API_KEY to .env")
    print("  Claude: add ANTHROPIC_API_KEY to .env")

Judges enabled: {'ollama': False, 'openai': True, 'claude': True}


In [4]:
pooled_df = pd.read_json("result/34_pooled_evaluation_set/pooled_candidates.json")
pooled_df = pooled_df[pooled_df["summary_trustworthy"]].reset_index(drop=True)
print(f"Trustworthy pooled candidates available for judging: {len(pooled_df)}")

if PILOT_N_QUERIES is not None:
    pilot_query_ids = sorted(pooled_df["query_id"].unique())[:PILOT_N_QUERIES]
    judge_df = pooled_df[pooled_df["query_id"].isin(pilot_query_ids)].reset_index(drop=True)
    print(f"PILOT MODE: judging {len(judge_df)} candidates across {PILOT_N_QUERIES} queries")
else:
    judge_df = pooled_df
    print(f"FULL RUN: judging all {len(judge_df)} candidates")

Trustworthy pooled candidates available for judging: 7822
PILOT MODE: judging 430 candidates across 5 queries


In [3]:
JUDGE_PROMPT_TEMPLATE = """You are judging search result relevance for a company search engine.

Search query: "{query}"

Candidate company:
Name: {name}
Country: {country}
Summary: {summary}

Rate how relevant this company is to the search query, using exactly one of these labels:
2 = highly relevant (a strong, direct match for the query)
1 = partially relevant (related but not a strong direct match)
0 = not relevant

Respond with ONLY a JSON object: {{"label": <0, 1, or 2>, "reason": "<one short sentence>"}}"""


def build_prompt(row):
    return JUDGE_PROMPT_TEMPLATE.format(
        query=row["query"], name=row["name"], country=row["country"], summary=row["summary"],
    )


def parse_judge_reply(text):
    try:
        start, end = text.index("{"), text.rindex("}") + 1
        parsed = json.loads(text[start:end])
        return int(parsed["label"]), parsed.get("reason", "")
    except (ValueError, KeyError, json.JSONDecodeError):
        return None, f"UNPARSEABLE: {text[:200]}"

In [4]:
def judge_ollama(prompt):
    resp = requests.post(
        f"{OLLAMA_BASE_URL}/api/generate",
        json={"model": OLLAMA_MODEL, "prompt": prompt, "stream": False},
        timeout=60,
    )
    resp.raise_for_status()
    return parse_judge_reply(resp.json()["response"])


def judge_openai(prompt):
    resp = requests.post(
        "https://api.openai.com/v1/chat/completions",
        headers={"Authorization": f"Bearer {OPENAI_API_KEY}", "Content-Type": "application/json"},
        json={"model": "gpt-4o-mini", "messages": [{"role": "user", "content": prompt}], "temperature": 0},
        timeout=60,
    )
    resp.raise_for_status()
    return parse_judge_reply(resp.json()["choices"][0]["message"]["content"])


def judge_claude(prompt):
    resp = requests.post(
        "https://api.anthropic.com/v1/messages",
        headers={"x-api-key": ANTHROPIC_API_KEY, "anthropic-version": "2023-06-01", "Content-Type": "application/json"},
        json={"model": "claude-haiku-4-5-20251001", "max_tokens": 200, "messages": [{"role": "user", "content": prompt}]},
        timeout=60,
    )
    resp.raise_for_status()
    return parse_judge_reply(resp.json()["content"][0]["text"])


JUDGE_FUNCS = {"ollama": judge_ollama, "openai": judge_openai, "claude": judge_claude}
ACTIVE_JUDGES = [name for name, enabled in ENABLED_JUDGES.items() if enabled]
print(f"Active judges this run: {ACTIVE_JUDGES}")

Active judges this run: ['openai', 'claude']


In [5]:
def load_cache():
    if CACHE_PATH.exists():
        return json.load(open(CACHE_PATH))
    return {}


def save_cache(cache):
    tmp_path = CACHE_PATH.with_suffix(".json.tmp")
    json.dump(cache, open(tmp_path, "w"), indent=2, default=str)
    tmp_path.replace(CACHE_PATH)


cache = load_cache()
print(f"Loaded {len(cache)} already-judged candidates from cache")

for i, row in judge_df.iterrows():
    key = f'{row["query_id"]}::{row["domain"]}'
    entry = cache.get(key, {})
    prompt = build_prompt(row)

    changed = False
    for judge_name in ACTIVE_JUDGES:
        if judge_name in entry:
            continue  # already judged by this one, don't re-spend money on it
        try:
            label, reason = JUDGE_FUNCS[judge_name](prompt)
        except requests.exceptions.RequestException as e:
            print(f"  [{judge_name}] error on {row['domain']}: {e}")
            continue
        entry[judge_name] = {"label": label, "reason": reason}
        changed = True

    if changed:
        cache[key] = entry
        save_cache(cache)  # save after EVERY candidate -- paid calls, never lose progress

    if (i + 1) % 25 == 0 or (i + 1) == len(judge_df):
        print(f"  {i+1}/{len(judge_df)} candidates processed")
    time.sleep(0.2)

print("Done for this run (or stopped early -- nothing already judged is lost, rerun to resume).")

Loaded 0 already-judged candidates from cache
  25/430 candidates processed
  50/430 candidates processed
  75/430 candidates processed
  100/430 candidates processed
  125/430 candidates processed
  150/430 candidates processed
  175/430 candidates processed
  200/430 candidates processed
  225/430 candidates processed
  250/430 candidates processed
  275/430 candidates processed
  300/430 candidates processed
  325/430 candidates processed
  350/430 candidates processed
  375/430 candidates processed
  400/430 candidates processed
  425/430 candidates processed
  430/430 candidates processed
Done for this run (or stopped early -- nothing already judged is lost, rerun to resume).


In [6]:
cache = load_cache()
rows = []
for key, entry in cache.items():
    query_id, domain = key.split("::", 1)
    labels = {j: entry[j]["label"] for j in ACTIVE_JUDGES if j in entry and entry[j]["label"] is not None}
    if len(labels) < 2:
        continue  # need at least 2 judges to talk about agreement
    unanimous = len(set(labels.values())) == 1
    rows.append({
        "query_id": int(query_id), "domain": domain, **{f"label_{j}": v for j, v in labels.items()},
        "agree": unanimous, "gold_label": list(labels.values())[0] if unanimous else None,
    })

results_df = pd.DataFrame(rows)
gold_df = results_df[results_df["agree"]]
review_df = results_df[~results_df["agree"]]

gold_df.to_json(OUTPUT_DIR / "gold_labels_agreed.json", orient="records", indent=2)
review_df.to_json(OUTPUT_DIR / "needs_manual_review.json", orient="records", indent=2)

print(f"Judged so far: {len(results_df)}")
print(f"Agreed (gold): {len(gold_df)} ({100*len(gold_df)/len(results_df):.1f}%)" if len(results_df) else "No results yet")
print(f"Disagreed (needs your review): {len(review_df)}")

Judged so far: 430
Agreed (gold): 360 (83.7%)
Disagreed (needs your review): 70


In [7]:
cache = load_cache()
pooled_lookup = pooled_df.set_index(["query_id", "domain"])

review_rows = []
for key, entry in cache.items():
    query_id_str, domain = key.split("::", 1)
    query_id = int(query_id_str)
    labels = {j: entry[j]["label"] for j in ACTIVE_JUDGES if j in entry and entry[j]["label"] is not None}
    if len(labels) < 2 or len(set(labels.values())) == 1:
        continue  # only the disagreements need a human review row

    context = pooled_lookup.loc[(query_id, domain)]
    row = {
        "query_id": query_id, "query": context["query"], "domain": domain, "name": context["name"],
        "summary": context["summary"],
    }
    for judge_name in ACTIVE_JUDGES:
        if judge_name in entry:
            row[f"{judge_name}_label"] = entry[judge_name]["label"]
            row[f"{judge_name}_reason"] = entry[judge_name]["reason"]
    row["your_label"] = ""  # fill in with 0, 1, or 2 -- this is the tie-breaker
    review_rows.append(row)

review_queue_df = pd.DataFrame(review_rows)
review_queue_path = OUTPUT_DIR / "review_queue.csv"
review_queue_df.to_csv(review_queue_path, index=False)
print(f"Saved {len(review_queue_df)} disagreements to review -> {review_queue_path}")
print("Open this in Excel/Sheets/a text editor, read the query + summary + each judge's label and reason, and fill in the your_label column with 0, 1, or 2.")

Saved 70 disagreements to review -> result/35_llm_judge_ensemble/review_queue.csv
Open this in Excel/Sheets/a text editor, read the query + summary + each judge's label and reason, and fill in the your_label column with 0, 1, or 2.


In [5]:
# Merge the judge-agreed gold labels with your manually reviewed tie-breaks into one final gold-labeled dataset.
gold_df = pd.read_json(OUTPUT_DIR / "gold_labels_agreed.json")
review_final = pd.read_csv(OUTPUT_DIR / "review_queue.csv")

# Rows marked SKIP (insufficient evidence, unreachable sites) are excluded entirely, not guessed at.
resolved = review_final[review_final["your_label"].astype(str).str.upper() != "SKIP"].copy()
resolved["gold_label"] = resolved["your_label"].astype(int)
resolved["source"] = "human_reviewed"

agreed = gold_df[["query_id", "domain", "gold_label"]].copy()
agreed["source"] = "judges_agreed"

final_gold = pd.concat([agreed, resolved[["query_id", "domain", "gold_label", "source"]]], ignore_index=True)

# Re-attach name/summary for traceability -- useful later for the BERT distillation step.
final_gold = final_gold.merge(
    pooled_df[["query_id", "domain", "query", "name", "summary"]].drop_duplicates(subset=["query_id", "domain"]),
    on=["query_id", "domain"], how="left",
)

n_skipped = len(review_final) - len(resolved)
final_gold_path = OUTPUT_DIR / "final_gold_labels.json"
final_gold.to_json(final_gold_path, orient="records", indent=2)

print(f"Judge-agreed labels   : {len(agreed)}")
print(f"Human-reviewed labels : {len(resolved)}")
print(f"Skipped (insufficient evidence / unreachable): {n_skipped}")
print(f"Final gold-labeled dataset: {len(final_gold)} candidates -> {final_gold_path}")
print()
print("Label distribution:")
print(final_gold["gold_label"].value_counts().sort_index())

Judge-agreed labels   : 360
Human-reviewed labels : 59
Skipped (insufficient evidence / unreachable): 11
Final gold-labeled dataset: 419 candidates -> result/35_llm_judge_ensemble/final_gold_labels.json

Label distribution:
gold_label
0      8
1    119
2    292
Name: count, dtype: int64


In [ ]:
# Retroactive check: how well does Ollama agree with the gold labels we already settled on
# (360 judge-agreed + 59 human-reviewed)? Requires Ollama running (see the run.sh convention
# in the setup cell above) -- rerun the availability check first if you just started it.
ENABLED_JUDGES["ollama"] = ollama_available()
print("Ollama available:", ENABLED_JUDGES["ollama"])
if not ENABLED_JUDGES["ollama"]:
    print("Start Ollama the same way run.sh does, then rerun this cell.")
else:
    final_gold = pd.read_json(OUTPUT_DIR / "final_gold_labels.json")

    ollama_cache_path = OUTPUT_DIR / "ollama_retro_check.json"
    ollama_results = json.load(open(ollama_cache_path)) if ollama_cache_path.exists() else {}

    for i, row in final_gold.iterrows():
        key = f'{row["query_id"]}::{row["domain"]}'
        if key in ollama_results:
            continue
        prompt = build_prompt(row)
        try:
            label, reason = judge_ollama(prompt)
        except requests.exceptions.RequestException as e:
            print(f"  error on {row['domain']}: {e}")
            continue
        ollama_results[key] = {"label": label, "reason": reason}
        json.dump(ollama_results, open(ollama_cache_path, "w"), indent=2, default=str)  # save after every row -- local model, but still avoid redoing work
        if (i + 1) % 50 == 0 or (i + 1) == len(final_gold):
            print(f"  {i+1}/{len(final_gold)} judged by Ollama")

    final_gold["ollama_label"] = final_gold.apply(
        lambda r: ollama_results.get(f'{r["query_id"]}::{r["domain"]}', {}).get("label"), axis=1,
    )
    scored = final_gold.dropna(subset=["ollama_label"])
    agree_rate = (scored["ollama_label"] == scored["gold_label"]).mean()
    print()
    print(f"Ollama ({OLLAMA_MODEL}) judged: {len(scored)}/{len(final_gold)}")
    print(f"Ollama matched the existing gold label: {agree_rate:.1%}")
    print()
    print("Confusion (ollama_label - gold_label):")
    print((scored["ollama_label"] - scored["gold_label"]).value_counts().sort_index())